# Cell 1: Environment Setup - Mounting Google Drive




In [ ]:
# Mount Google Drive to access project files and datasets
from google.colab import drive

drive.mount('/content/drive')

# Cell 2: Advanced Environment Initialization

In [ ]:
import os
from google.colab import drive

print("System: Initializing compute environment...\n")

# Mount Google Drive with force_remount to prevent timeout errors
drive.mount('/content/drive', force_remount=True)

print("System: Google Drive mounted successfully.")

# Cell 3: Dataset Extraction to Local Storage

In [ ]:
import os

print("System: Verifying Google Drive connection...")

# Mount Drive if not already accessible
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

print("System: Extracting primary dataset to local storage for optimal performance...\n")

# Extract dataset to local storage for faster I/O
!unzip -q -o "/content/drive/MyDrive/Tumor_Project/archive.zip" -d "/content/archive"

print("System: High-speed local data extraction complete.")
print("System: Files are ready for processing.")

# Cell 4: Importing Global Dependencies

In [ ]:
import os
import numpy as np
from PIL import Image

# Machine learning evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

# PyTorch core, mixed precision, and data utilities
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

print("System: Global dependencies and libraries imported successfully.")

# Cell 5: Hardware Initialization and Reproducibility Setup

In [ ]:
print("System: Initializing hardware and securing runtime environment...\n")

# Assign hardware accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"System: Hardware compute initialized on [{device.type.upper()}].")

# Lock random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("System: Reproducibility locks engaged. Environment is stable.")

# Cell 6: Dataset Architectures and Transformations

In [ ]:
import os
import cv2
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset

print("System: Initializing dataset architectures and transformations...")

# Training transformations (includes augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Validation/testing transformations (strictly clean)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Global multimodal dataset class
class GlobalMultimodalDataset(Dataset):
    def __init__(self, root_dirs, transform=None):
        self.transform = transform
        self.samples = []
        self.class_map = {"healthy": 0, "notumor": 0, "tumor": 1, "tumour": 1}

        all_files = []
        for root_dir in root_dirs:
            if not os.path.exists(root_dir):
                print(f"System Warning: Directory {root_dir} not found. Skipping.")
                continue

            for subdir, _, files in os.walk(root_dir):
                modality = "ct" if "ct" in subdir.lower() else "mri"
                folder_name = os.path.basename(subdir).lower()

                label = None
                for key, val in self.class_map.items():
                    if key in folder_name:
                        label = val
                        break

                if label is None:
                    continue

                valid_files = [
                    os.path.join(subdir, f) for f in files
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp', '.dcm'))
                ]

                for f in valid_files:
                    all_files.append((f, label, modality))

        print("System: Loading all files into memory maps...")

        # Progress bar to track file ingestion
        progress_bar = tqdm(
            all_files,
            desc="Ingesting Entire Dataset",
            leave=True,
            bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt}'
        )

        for file_path, label, modality in progress_bar:
            self.samples.append({
                "path": file_path,
                "label": label,
                "modality": modality
            })

        print(f"\nSystem: Ingestion complete. Total samples retained: {len(self.samples)} (100% of dataset).")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = Image.open(sample["path"]).convert("RGB")
        label = torch.tensor(sample["label"], dtype=torch.float32)
        modality_flag = torch.tensor(1.0 if sample["modality"] == "mri" else 0.0, dtype=torch.float32)

        if self.transform:
            img = self.transform(img)

        return img, label, modality_flag

# Dynamic transform wrapper for dataset subsets
class DatasetTransformSplit(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label, mod_flag = self.subset[idx]
        if self.transform:
            img = self.transform(img)

        return img, label, mod_flag

# Cell 7: Dataset Compilation and Stratification Prep

In [ ]:
import numpy as np

print("System: Compiling global dataset from extracted archives...\n")

# Path to the local extracted archive
data_directory = "/content/archive"

# Instantiate global dataset using the unfiltered class
global_dataset = GlobalMultimodalDataset(root_dirs=[data_directory], transform=None)

# Generate composite labels (Label + Modality) for balanced K-Fold splitting
composite_labels = [
    f"{int(s['label'])}_{1 if s['modality'] == 'mri' else 0}"
    for s in global_dataset.samples
]
labels_array = np.array(composite_labels)

print(f"\nSystem: Global dataset compiled successfully. Total images ready: {len(global_dataset)}")
print("System: Data structure is perfectly prepped for the Stratified K-Fold engine.")

# Cell 8: Stratified K-Fold Cross-Validation Setup

In [ ]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset

print("System: Initializing Stratified 5-Fold Cross-Validation Splitter...\n")

# Cross-Validation setup
NUM_FOLDS = 5
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

folds_data = {}

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels_array)), labels_array)):
    print(f"System: Generating partition arrays for Fold {fold_idx + 1}...")

    # Construct subset slices from the global dataset
    raw_train_subset = Subset(global_dataset, train_idx)
    raw_val_subset = Subset(global_dataset, val_idx)

    # Bind subsets to train and validation transformations
    folds_data[fold_idx + 1] = {
        "train_dataset": DatasetTransformSplit(raw_train_subset, transform=train_transform),
        "val_dataset": DatasetTransformSplit(raw_val_subset, transform=test_transform)
    }

print(f"\nSystem: Partitions successfully stabilized across {NUM_FOLDS} folds.")
print("System: Downstream training and metrics evaluation blocks are ready to execute.")

# Cell 9: Advanced Objective Functions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

print("System: Initializing advanced objective functions...")

class FocalLoss(nn.Module):
    """
    Addresses class imbalance by down-weighting well-classified examples
    and focusing training on hard negatives.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        targets = targets.view_as(inputs)
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss
        return torch.mean(F_loss)

class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss. Pulls samples of the same class closer
    in the latent space while pushing different classes apart.
    """
    def __init__(self, temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        anchor_dot_contrast = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(features.shape[0]).view(-1, 1).to(device), 0)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-8)

        mask_sum = mask.sum(1)
        mask_sum = torch.where(mask_sum == 0, torch.ones_like(mask_sum), mask_sum)

        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_sum
        return -mean_log_prob_pos.mean()

# Cell 10: Vision Mamba Architectural Blocks

In [ ]:
import torch
import torch.nn as nn

print("System: Compiling Vision Mamba architectural blocks...")

class PyTorchNativeVimBlock(nn.Module):
    """
    Native PyTorch implementation of the Vision Mamba (Vim) block.
    Utilizes a bidirectional GRU as an SSM proxy for sequence modeling.
    """
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.proj1 = nn.Linear(d_model, d_model * 2)
        self.conv = nn.Conv1d(d_model * 2, d_model * 2, kernel_size=3, padding=1, groups=d_model * 2)
        self.ssm_proxy = nn.GRU(d_model * 2, d_model, batch_first=True, bidirectional=True)
        self.proj2 = nn.Linear(d_model * 2, d_model)
        self.silu = nn.SiLU()

    def forward(self, x):
        res = x
        x = self.norm(x)
        x = self.proj1(x)
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        x = self.silu(x)
        x, _ = self.ssm_proxy(x)
        x = self.proj2(x)
        return x + res

class VisionMambaExtractor(nn.Module):
    """
    Translates 2D clinical scans into 1D patch sequences and processes them
    through successive Vision Mamba blocks.
    """
    def __init__(self, img_size=224, patch_size=8, in_chans=3, embed_dim=256, depth=3):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.blocks = nn.ModuleList([PyTorchNativeVimBlock(embed_dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.feature_map = nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.patch_embed(x)
        spatial_H, spatial_W = x.shape[2], x.shape[3]
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        x = x.transpose(1, 2).reshape(B, -1, spatial_H, spatial_W)
        x = self.feature_map(x)
        return x

# Cell 11: Prototype Cross-Attention Networks

In [ ]:
import torch
import torch.nn as nn

print("System: Compiling Prototype Cross-Attention networks...")

class PrototypeCrossAttentionFusion(nn.Module):
    """
    Computes cross-attention between spatial feature sequences and learned clinical prototypes.
    """
    def __init__(self, feature_dim=128, num_heads=4):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(feature_dim)

    def forward(self, spatial_sequence, prototypes):
        B = spatial_sequence.size(0)
        Q = prototypes.unsqueeze(0).expand(B, -1, -1)
        attended_features, attn_weights = self.cross_attn(query=Q, key=spatial_sequence, value=spatial_sequence)
        fused_features = self.norm(Q + attended_features)
        return fused_features, attn_weights

class PrototypeClassifier(nn.Module):
    """
    Calculates the final classification logits based on L2 distance from learned prototypes.
    """
    def __init__(self, feature_dim=128, num_classes=2):
        super().__init__()
        self.prototypes = nn.Parameter(torch.randn(num_classes, feature_dim))

    def forward(self, fused_features):
        diff = fused_features - self.prototypes.unsqueeze(0)
        distances = (diff ** 2).sum(dim=2)
        dist_healthy = distances[:, 0:1]
        dist_tumor = distances[:, 1:2]
        logits = dist_healthy - dist_tumor
        return logits

# Cell 12: Assembling Global Multimodal Architecture

In [ ]:
print("System: Assembling global multimodal architecture...\n")

class UnpairedMultimodalModel(nn.Module):
    """
    The master architecture. Routes data through respective modality branches,
    projects features into a unified latent space, and executes prototype fusion.
    """
    def __init__(self):
        super().__init__()
        self.mri_branch = VisionMambaExtractor(img_size=224, patch_size=8, embed_dim=256, depth=3)
        self.ct_branch = VisionMambaExtractor(img_size=224, patch_size=8, embed_dim=256, depth=3)
        self.projection_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )
        self.prototype_classifier = PrototypeClassifier(feature_dim=128, num_classes=2)
        self.fusion_block = PrototypeCrossAttentionFusion(feature_dim=128, num_heads=4)

    def forward(self, x, modality_flags):
        batch_size = x.size(0)
        output_features = torch.zeros(batch_size, 256, x.size(2)//8, x.size(3)//8, device=x.device, dtype=x.dtype)

        mri_mask = (modality_flags == 1.0)
        if mri_mask.any():
            output_features[mri_mask] = self.mri_branch(x[mri_mask]).to(output_features.dtype)

        ct_mask = (modality_flags == 0.0)
        if ct_mask.any():
            output_features[ct_mask] = self.ct_branch(x[ct_mask]).to(output_features.dtype)

        B, C, H, W = output_features.shape
        spatial_sequence = output_features.view(B, C, -1).transpose(1, 2)
        projected_sequence = F.normalize(self.projection_head(spatial_sequence), dim=2)
        global_features = projected_sequence.mean(dim=1)

        prototypes = F.normalize(self.prototype_classifier.prototypes, dim=1)
        fused_features, attention_maps = self.fusion_block(projected_sequence, prototypes)
        logits = self.prototype_classifier(fused_features)

        return logits, global_features, (output_features, attention_maps)

def build_model_pipeline(epochs=20):
    """
    Spawns isolated model instances alongside specific optimizers, schedulers,
    and mixed-precision scalers for K-Fold evaluation.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UnpairedMultimodalModel().to(device)

    criterion_focal = FocalLoss(alpha=0.25, gamma=2.0)
    criterion_supcon = SupConLoss(temperature=0.07)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Explicit device keyword argument for newer PyTorch AMP API compatibility
    scaler = GradScaler(device='cuda' if torch.cuda.is_available() else 'cpu')

    return device, model, criterion_focal, criterion_supcon, optimizer, scheduler, scaler

print("System: Model Factory initialized. Ready to spawn isolated K-Fold environments.")

# Cell 13: Evaluation Metrics Engine

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    matthews_corrcoef, cohen_kappa_score
)

def evaluate_performance_metrics(y_true, y_pred, y_probs):
    """
    Computes standard classification metrics alongside Specificity,
    MCC, and Cohen's Kappa extracted from the Confusion Matrix.
    """
    # 1. Standard Metrics (Zero division handling for stability)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    sensitivity = recall_score(y_true, y_pred, zero_division=0) # Recall = Sensitivity
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # 2. ROC-AUC (Requires raw probabilities)
    try:
        auc = roc_auc_score(y_true, y_probs)
    except ValueError:
        auc = 0.5 # Safety fallback if a batch contains only one class

    # 3. Confusion Matrix & Specificity Extraction
    # Unravel the matrix into True Negatives, False Positives, False Negatives, True Positives
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    # Calculate Specificity (TN Rate). Epsilon (1e-9) prevents division by zero.
    specificity = tn / (tn + fp + 1e-9)

    # 4. Advanced Clinical Metrics
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    print("\nSystem: Validation Performance Metrics Compiled\n")
    print("--- Confusion Matrix ---")
    print(f"                Predicted Predicted ")
    print(f"                 Healthy | Tumor")
    print(f"Actual Healthy | {tn:<7} | {fp:<5}")
    print(f"Actual Tumor   | {fn:<7} | {tp:<5}")
    print("------------------------")
    print(f"Accuracy         : {accuracy:.4f}")
    print(f"Precision        : {precision:.4f}")
    print(f"Sensitivity      : {sensitivity:.4f}")
    print(f"Specificity      : {specificity:.4f}")
    print(f"F1-Score         : {f1:.4f}")
    print(f"ROC-AUC          : {auc:.4f}")
    print(f"MCC              : {mcc:.4f}")
    print(f"Cohen Kappa      : {kappa:.4f}\n")

    # Return a dictionary so your training loop can track and plot these over time
    return {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1_score": f1,
        "roc_auc": auc,
        "mcc": mcc,
        "cohen_kappa": kappa,
        "confusion_matrix": cm
    }

print("System: Advanced Metrics Engine initialized.")
print("System: Model performance tracking is ready.")

# Cell 14: K-Fold Express Cross-Validation Setup

In [ ]:
import os
import copy
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from google.colab import drive

print("System: Verifying persistent cloud storage connection...\n")

# Ensure Drive is connected to secure model weights
try:
    drive.mount('/content/drive')
    print("System: Google Drive connection verified.")
except Exception as e:
    print(f"System Warning: Drive connection failed. {e}")

# High-Speed Training Hyperparameters
K_FOLDS = len(folds_data)  # Dynamically pulls pre-stratified folds
EPOCHS = 12                # Optimized epoch count for rapid iteration
ALPHA_SUPCON = 0.1         # Weighting factor for Supervised Contrastive Loss
PATIENCE = 3               # Aggressive early stopping threshold

print(f"\nSystem: Initiating {K_FOLDS}-Fold Express Cross-Validation.")

# Cell 15: K-Fold Execution & Auto-Save Protocol

In [ ]:
import os
import copy
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# Execution wrapper for our Metrics Engine
def evaluate_model(eval_model, loader, dataset_name="EVALUATION"):
    print(f"\nSystem: Running inference on {dataset_name}...")
    eval_model.eval()

    all_targets = []
    all_probs = []
    all_preds = []

    # Dynamically detect which device the model is living on
    current_device = next(eval_model.parameters()).device

    with torch.no_grad():
        for val_images, val_labels, val_mod_flags in loader:
            val_images = val_images.to(current_device)
            val_labels = val_labels.to(current_device)
            val_mod_flags = val_mod_flags.to(current_device)

            with torch.amp.autocast(device_type='cuda'):
                val_logits, _, _ = eval_model(val_images, val_mod_flags)
                # Convert raw network logits into percentages (0.0 to 1.0)
                probs = torch.sigmoid(val_logits).squeeze()

            # Safety catch: If the final batch has only 1 image, squeeze() makes it a 0-d tensor
            if probs.dim() == 0:
                probs = probs.unsqueeze(0)

            # Convert probabilities to hard binary predictions (0 or 1)
            preds = (probs > 0.5).int()

            all_targets.extend(val_labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    metrics = evaluate_performance_metrics(all_targets, all_preds, all_probs)
    return metrics

# MAIN TRAINING LOOP
for fold_num, fold_datasets in folds_data.items():
    print(f"System: Launching Pipeline for Fold {fold_num}/{K_FOLDS}\n")

    # Hardware Optimization: High batch size to saturate GPU processing
    train_loader = DataLoader(fold_datasets["train_dataset"], batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(fold_datasets["val_dataset"], batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    print(f" Allocated Training Scans:   {len(fold_datasets['train_dataset'])}")
    print(f" Allocated Validation Scans: {len(fold_datasets['val_dataset'])}")

    # Initialize a pristine neural network for the current fold
    device, model, criterion_focal, criterion_supcon, optimizer, scheduler, scaler = build_model_pipeline(epochs=EPOCHS)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_weights = None

    # Training Phase
    for epoch in range(EPOCHS):
        model.train()
        train_epoch_loss = 0

        # Progress bar
        batch_iterator = tqdm(
            train_loader,
            desc=f"Fold {fold_num} | Epoch [{epoch+1:02d}/{EPOCHS}] [TRAIN]",
            leave=False,
            bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt}'
        )

        for images, labels, modality_flags in batch_iterator:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            modality_flags = modality_flags.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type='cuda'):
                logits, projected_features, _ = model(images, modality_flags)
                loss_cls = criterion_focal(logits, labels)
                loss_supcon = criterion_supcon(projected_features, labels)
                loss = loss_cls + (ALPHA_SUPCON * loss_supcon)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_epoch_loss += loss.item()
            batch_iterator.set_postfix(train_loss=f"{loss.item():.4f}")

        scheduler.step()
        avg_train_loss = train_epoch_loss / len(train_loader)

        # Validation Phase
        model.eval()
        val_epoch_loss = 0

        with torch.no_grad():
            for val_images, val_labels, val_mod_flags in test_loader:
                val_images = val_images.to(device)
                val_labels = val_labels.to(device).unsqueeze(1)
                val_mod_flags = val_mod_flags.to(device)

                with torch.amp.autocast(device_type='cuda'):
                    val_logits, val_projected, _ = model(val_images, val_mod_flags)
                    v_loss_cls = criterion_focal(val_logits, val_labels)
                    v_loss_supcon = criterion_supcon(val_projected, val_labels)
                    v_loss = v_loss_cls + (ALPHA_SUPCON * v_loss_supcon)

                val_epoch_loss += v_loss.item()

        avg_val_loss = val_epoch_loss / len(test_loader)

        print(f"Fold {fold_num} | Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # Early Stopping & Checkpoint Logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            best_model_weights = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f" System Notice: Early stopping criteria met at Epoch {epoch+1}.")
                break

    # Security Protocol: Persist best model state to Drive
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nSystem: Optimal internal weights restored for Fold {fold_num}.")

        save_dir = "/content/drive/MyDrive/Tumor_Project"
        os.makedirs(save_dir, exist_ok=True)
        fold_path = os.path.join(save_dir, f"VisionMamba_Fold_{fold_num}_Best.pth")

        torch.save(model.state_dict(), fold_path)
        print(f"System: Security checkpoint created. Weights secured at: {fold_path}")

    # Generate immediate diagnostic report for the current fold
    evaluate_model(model, test_loader, dataset_name=f"FOLD {fold_num} UNSEEN TESTING")

print(f"\nSystem: {K_FOLDS}-Fold Cross-Validation Protocol Successfully Completed.\n")

# Cell 16: Test-Time Augmentation (TTA) Evaluation Engine

In [ ]:
import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, cohen_kappa_score
)

print("System: Initializing Test-Time Augmentation (TTA) evaluation engine...")

def evaluate_with_tta(model, dataloader, device, use_tta=False):
    """
    Evaluates the model. If use_tta is enabled, it applies Test-Time Augmentation
    by calculating the consensus average across original, horizontally flipped,
    and vertically flipped images.
    """
    model.eval()
    all_targets, all_probs = [], []

    with torch.no_grad():
        for images, labels, modality_flags in dataloader:
            images = images.to(device)
            modality_flags = modality_flags.to(device)
            all_targets.extend(labels.numpy().flatten())

            with torch.amp.autocast(device_type='cuda'):
                logits_orig, _, _ = model(images, modality_flags)
                probs_orig = torch.sigmoid(logits_orig)

                if use_tta:
                    # Pass 2: Horizontal Flip
                    images_hflip = torch.flip(images, dims=[3])
                    logits_hflip, _, _ = model(images_hflip, modality_flags)
                    probs_hflip = torch.sigmoid(logits_hflip)

                    # Pass 3: Vertical Flip
                    images_vflip = torch.flip(images, dims=[2])
                    logits_vflip, _, _ = model(images_vflip, modality_flags)
                    probs_vflip = torch.sigmoid(logits_vflip)

                    # Consensus average probability
                    final_probs = (probs_orig + probs_hflip + probs_vflip) / 3.0
                else:
                    final_probs = probs_orig

            all_probs.extend(final_probs.cpu().numpy().flatten())

    # Format predictions for sklearn compatibility
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)
    all_preds = (all_probs > 0.5).astype(int)

    return {
        "Accuracy": accuracy_score(all_targets, all_preds),
        "ROC-AUC": roc_auc_score(all_targets, all_probs),
        "Precision": precision_score(all_targets, all_preds, zero_division=0),
        "Recall (Sensitivity)": recall_score(all_targets, all_preds, zero_division=0),
        "F1-Score": f1_score(all_targets, all_preds, zero_division=0),
        "Matthews Corrcoef (MCC)": matthews_corrcoef(all_targets, all_preds),
        "Cohen Kappa": cohen_kappa_score(all_targets, all_preds)
    }

# Cell 17: Final Aggregate Cross-Validation Report

In [ ]:
import os
import io
import torch
import numpy as np
from contextlib import redirect_stdout
from torch.utils.data import DataLoader

print("System: Compiling final aggregate cross-validation report...\n")

# Initialize memory arrays for the core clinical metrics
metric_keys = ["accuracy", "precision", "sensitivity", "specificity", "f1_score", "roc_auc", "mcc", "cohen_kappa"]
display_names = ["Accuracy", "Precision", "Sensitivity", "Specificity", "F1-Score", "ROC-AUC", "MCC", "Cohen Kappa"]

aggregate_train = {key: [] for key in metric_keys}
aggregate_test = {key: [] for key in metric_keys}
save_dir = "/content/drive/MyDrive/Tumor_Project"

for fold_idx in folds_data.keys():
    print(f"System: Extracting diagnostics from Fold {fold_idx}...")

    # Apples-to-Apples Clean Evaluation Patch
    raw_train_subset = folds_data[fold_idx]["train_dataset"].subset
    clean_train_dataset = DatasetTransformSplit(raw_train_subset, transform=test_transform)
    eval_train_loader = DataLoader(clean_train_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    fold_datasets = folds_data[fold_idx]
    eval_test_loader = DataLoader(fold_datasets["val_dataset"], batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    # Initialize model and load secured weights
    device, eval_model, _, _, _, _, _ = build_model_pipeline(epochs=1)
    fold_path = os.path.join(save_dir, f"VisionMamba_Fold_{fold_idx}_Best.pth")

    try:
        eval_model.load_state_dict(torch.load(fold_path, map_location=device))
        eval_model.to(device)
    except FileNotFoundError:
        print(f" Error: Could not find checkpoint for Fold {fold_idx}. Skipping...")
        continue

    # Temporarily block all print statements from the inner evaluation functions
    with redirect_stdout(io.StringIO()):
        train_metrics = evaluate_model(eval_model, eval_train_loader, dataset_name=f"FOLD {fold_idx} CLEAN TRAIN")
        test_metrics = evaluate_model(eval_model, eval_test_loader, dataset_name=f"FOLD {fold_idx} UNSEEN TEST")

    # Append results to memory arrays
    for key in metric_keys:
        aggregate_train[key].append(train_metrics[key])
        aggregate_test[key].append(test_metrics[key])

def print_aggregate_table(agg_metrics, title):
    print("\n=======================================================")
    print(title)
    print("=======================================================")
    print(f"{'Performance Metric':<15} | {'Average (Mean ± Std)':<20} | {'Best Score'}")
    print("-------------------------------------------------------")

    for key, display_name in zip(metric_keys, display_names):
        values = agg_metrics[key]
        if not values: continue # Skip if no data

        mean_val = np.mean(values)
        std_val = np.std(values)
        best_val = np.max(values)

        print(f"{display_name:<15}    | {mean_val:.4f} ± {std_val:.4f}      | {best_val:.4f}")
    print("=======================================================")

def print_difference_table(train_agg, test_agg, title):
    print("\n=======================================================")
    print(title)
    print("=======================================================")
    print(f"{'Performance Metric':<15} | {'Train Mean':<12} | {'Test Mean':<12} | {'Gap (Train - Test)'}")
    print("-------------------------------------------------------")

    for key, display_name in zip(metric_keys, display_names):
        train_vals = train_agg[key]
        test_vals = test_agg[key]
        if not train_vals or not test_vals: continue

        train_mean = np.mean(train_vals)
        test_mean = np.mean(test_vals)
        diff = train_mean - test_mean

        # The :+.4f format automatically adds a + sign for positive numbers
        print(f"{display_name:<15}    | {train_mean:.4f}       | {test_mean:.4f}       | {diff:+.4f}")
    print("=======================================================\n")

# Display final outputs (Prints restored automatically outside the 'with' block)
print_aggregate_table(aggregate_train, "FINAL AGGREGATE: TRAINING DATA (Clean Evaluation)")
print_aggregate_table(aggregate_test, "FINAL AGGREGATE: UNSEEN TESTING DATA")
print_difference_table(aggregate_train, aggregate_test, "GENERALIZATION GAP: TRAIN VS. TEST DIFFERENCE")

print("System: Clinical Pipeline Evaluation Complete.")

# Cell 18: Image Processing and Visualization Utilities

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

print("System: Loading image processing and visualization utilities...")

def unnormalize_image(img_tensor):
    """
    Reverses the normalization transformations applied during the data ingestion phase.
    This ensures the scans display accurately for human visual review without color distortion.
    """
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)

    # Reverse the standardized Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) parameter
    img = (img * 0.5) + 0.5

    return np.clip(img, 0, 1)

# Cell 19: Advanced Explainable AI (XAI) Rendering Engine

In [ ]:
print("System: Compiling Advanced Explainable AI (XAI) Rendering Engine...")
print("System: Modules included GradCAM++, SSM Attention, Prototype Similarity With Scores")

def render_clinical_explanations(model, dataloader, device, num_samples=4):

    # Initialize Advanced XAI engines
    grad_cam_pp_engine = DualBranchGradCAMPlusPlus(model)
    ssm_attn_engine = VisionMambaAttentionExtractor(model)
    prototype_engine = PrototypeSimilarityMap(model)

    # Extract a single randomized batch of unseen test data
    images, labels, mod_flags = next(iter(dataloader))
    images = images.to(device)
    mod_flags = mod_flags.to(device)
    labels = labels.to(device)

    # Initialize the standard clinical viewing grid
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5 * num_samples))
    if num_samples == 1: axes = [axes]

    for i in range(num_samples):
        img_tensor = images[i:i+1]
        mod_flag = mod_flags[i:i+1]
        true_label = int(labels[i].item())

        # 1. Generate core XAI Heatmaps & Scores
        cam_heatmap, prob = grad_cam_pp_engine.generate_heatmap(img_tensor, mod_flag)
        attn_heatmap = ssm_attn_engine.generate_heatmap(img_tensor, mod_flag)

        # Unpack the new similarity score alongside the heatmap
        sim_heatmap, sim_score = prototype_engine.generate_heatmap(img_tensor, mod_flag)

        # 2. Format Clinical Verdict String
        pred_class = 1 if prob > 0.5 else 0
        match_status = "[MATCH]" if pred_class == true_label else "[ERROR]"
        true_str = "Tumor" if true_label == 1 else "Healthy"
        pred_str = "Tumor" if pred_class == 1 else "Healthy"
        title_str = f"True: {true_str} | Pred: {pred_str} ({prob:.2f}) {match_status}"

        # 3. Process Original Image
        orig_img = unnormalize_image(images[i])

        # 4. Resize and Overlay Methods
        def apply_overlay(heatmap, colormap):
            resized = cv2.resize(heatmap, (224, 224))
            color = cv2.applyColorMap(np.uint8(255 * resized), colormap)
            color = np.float32(color[:, :, ::-1]) / 255
            return np.clip((color * 0.5) + (orig_img * 0.5), 0, 1)

        cam_overlay = apply_overlay(cam_heatmap, cv2.COLORMAP_JET)
        attn_overlay = apply_overlay(attn_heatmap, cv2.COLORMAP_VIRIDIS)
        sim_overlay = apply_overlay(sim_heatmap, cv2.COLORMAP_MAGMA)

        # --- PLOTTING ARCHITECTURE ---
        axes[i][0].imshow(orig_img)
        axes[i][0].set_title(f"Original Scan\n{title_str}", fontweight="bold")
        axes[i][0].axis('off')

        axes[i][1].imshow(cam_overlay)
        axes[i][1].set_title("Grad-CAM++\n(Sharp Feature Gradient)", fontweight="bold")
        axes[i][1].axis('off')

        axes[i][2].imshow(attn_overlay)
        axes[i][2].set_title("SSM Attention\n(Mamba Activation Footprint)", fontweight="bold")
        axes[i][2].axis('off')

        axes[i][3].imshow(sim_overlay)
        # Display the quantitative score in the title
        axes[i][3].set_title(f"Cross-Attention\n(Prototype Score: {sim_score:.4f})", fontweight="bold")
        axes[i][3].axis('off')

    plt.tight_layout()
    plt.show()

# Cell 20: Standalone Advanced Explainable AI (XAI) Engines

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

print("System: Compiling standalone Advanced Explainable AI (XAI) engines...")

class DualBranchGradCAMPlusPlus:
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        self.model.mri_branch.feature_map.register_forward_hook(self.save_activation)
        self.model.mri_branch.feature_map.register_full_backward_hook(self.save_gradient)
        self.model.ct_branch.feature_map.register_forward_hook(self.save_activation)
        self.model.ct_branch.feature_map.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_heatmap(self, input_image, modality_flag):
        self.model.train()
        self.model.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            logits, _, _ = self.model(input_image, modality_flag)

        prob = torch.sigmoid(logits)
        logits.sum().backward(retain_graph=True, create_graph=True)

        gradients = self.gradients.detach()
        activations = self.activations.detach()

        alpha_num = gradients.pow(2)
        alpha_denom = gradients.pow(2).mul(2) + activations.mul(gradients.pow(3)).sum(dim=[2, 3], keepdim=True)
        alpha_denom = torch.where(alpha_denom != 0.0, alpha_denom, torch.ones_like(alpha_denom))
        alphas = alpha_num / alpha_denom

        weights = (alphas * torch.relu(gradients)).sum(dim=[2, 3], keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)[0, 0].cpu().numpy()

        if np.max(cam) != 0:
            cam /= np.max(cam)

        self.model.eval()
        return cam, prob.item()


class VisionMambaAttentionExtractor:
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.model.mri_branch.blocks[-1].silu.register_forward_hook(self.save_activation)
        self.model.ct_branch.blocks[-1].silu.register_forward_hook(self.save_activation)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def generate_heatmap(self, input_image, modality_flag):
        self.model.eval()
        with torch.no_grad():
            with torch.amp.autocast(device_type='cuda'):
                _ = self.model(input_image, modality_flag)

        activation_mag = torch.norm(self.activations[0], dim=-1)
        grid_size = int(np.sqrt(activation_mag.shape[0]))

        attention_map = activation_mag.view(grid_size, grid_size).cpu().numpy()
        attention_map = attention_map - np.min(attention_map)
        if np.max(attention_map) != 0:
            attention_map /= np.max(attention_map)

        return attention_map


class PrototypeSimilarityMap:
    def __init__(self, model):
        self.model = model

    def generate_heatmap(self, input_image, modality_flag):
        self.model.eval()
        with torch.no_grad():
            with torch.amp.autocast(device_type='cuda'):
                logits, _, (_, attention_maps) = self.model(input_image, modality_flag)

                _, num_classes, num_patches = attention_maps.shape
                grid_size = int(np.sqrt(num_patches))
                prob = torch.sigmoid(logits).item()
                pred_class = 1 if prob > 0.5 else 0

                # Extract the attention map for the predicted class prototype
                target_attention = attention_maps[0, pred_class, :]

        # 1. Capture the raw similarity map
        similarity_map_raw = target_attention.view(grid_size, grid_size).cpu().numpy()

        # 2. Extract the Maximum Similarity Score before normalization
        # This represents the strongest "hotspot" match against the clinical prototype
        sim_score = float(np.max(similarity_map_raw))

        # 3. Min-Max normalize for standard image plotting
        similarity_map = similarity_map_raw - np.min(similarity_map_raw)
        if np.max(similarity_map) != 0:
            similarity_map /= np.max(similarity_map)

        return similarity_map, sim_score

# Cell 21: Multi-colormap Clinical Legend Module

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("System: Loading multi-colormap clinical legend module...")

def plot_clinical_legend():

    fig, axes = plt.subplots(3, 1, figsize=(18, 3.5))
    gradient = np.linspace(0, 1, 256)
    gradient = np.vstack((gradient, gradient))

    # Grad-CAM++ Legend (JET)
    axes[0].imshow(gradient, aspect='auto', cmap='jet')
    axes[0].set_title("Grad-CAM++ (JET): Deep Blue = 0% | Cyan/Green = 50% | Deep Red = 100% Hotspot", loc='center', fontweight='bold', fontsize=10)
    axes[0].axis('off')

    # SSM Attention Legend (VIRIDIS)
    axes[1].imshow(gradient, aspect='auto', cmap='viridis')
    axes[1].set_title("Mamba SSM Attention (VIRIDIS): Dark Purple = Low Activity | Green = Medium | Yellow = Peak Sequence Activation", loc='center', fontweight='bold', fontsize=10)
    axes[1].axis('off')

    # Prototype Legend (MAGMA)
    axes[2].imshow(gradient, aspect='auto', cmap='magma')
    axes[2].set_title("Prototype Similarity (MAGMA): Black = 0% Match | Orange = Strong Match | White = 100% Exact Match", loc='center', fontweight='bold', fontsize=10)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Cell 22: Mass Validation Gallery Execution

In [ ]:
import cv2
import warnings
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

# FIX: Globally silence PyTorch computational graph warnings for clean rendering
warnings.filterwarnings('ignore')

print("System: Compiling Advanced Mass Validation Gallery (20 Unseen Samples)...\n")

try:
    gallery_loader = DataLoader(folds_data[1]["val_dataset"], batch_size=20, shuffle=True)
    images, labels, mod_flags = next(iter(gallery_loader))

    images = images.to(device)
    labels = labels.to(device)
    mod_flags = mod_flags.to(device)

    cam_engine = DualBranchGradCAMPlusPlus(eval_model)
    attn_engine = VisionMambaAttentionExtractor(eval_model)
    proto_engine = PrototypeSimilarityMap(eval_model)

    chunks = 5
    samples_per_chunk = 4

    print("System: Rendering visual legend...")
    plot_clinical_legend()
    print("--------------------------------------------------\n")

    for chunk in range(chunks):
        fig, axes = plt.subplots(samples_per_chunk, 5, figsize=(22, 5 * samples_per_chunk))

        for i in range(samples_per_chunk):
            idx = (chunk * samples_per_chunk) + i

            img_tensor = images[idx:idx+1]
            mod_flag = mod_flags[idx:idx+1]
            true_label = int(labels[idx].item())

            cam_heatmap, prob = cam_engine.generate_heatmap(img_tensor, mod_flag)
            attn_heatmap = attn_engine.generate_heatmap(img_tensor, mod_flag)

            # Extract both heatmap and numerical score
            sim_heatmap, sim_score = proto_engine.generate_heatmap(img_tensor, mod_flag)

            pred_class = 1 if prob > 0.5 else 0
            match_icon = "[MATCH]" if pred_class == true_label else "[ERROR]"
            true_str = "TUMOR" if true_label == 1 else "HEALTHY"
            pred_str = "TUMOR" if pred_class == 1 else "HEALTHY"
            mod_str = "MRI" if mod_flag.item() == 1.0 else "CT"
            confidence = prob * 100 if prob > 0.5 else (1 - prob) * 100

            orig_img = img_tensor[0].clone().detach().cpu().numpy().transpose(1, 2, 0)
            orig_img = np.clip((orig_img * 0.5) + 0.5, 0, 1)

            def get_overlay(heatmap, cmap):
                # FORCE FLOAT32: Prevents OpenCV float16 assertion crash
                heatmap = np.float32(heatmap)

                resized = cv2.resize(heatmap, (224, 224))
                color = cv2.applyColorMap(np.uint8(255 * resized), cmap)
                color = np.float32(color[:, :, ::-1]) / 255
                return np.clip((color * 0.5) + (orig_img * 0.5), 0, 1)

            cam_overlay = get_overlay(cam_heatmap, cv2.COLORMAP_JET)
            attn_overlay = get_overlay(attn_heatmap, cv2.COLORMAP_VIRIDIS)
            sim_overlay = get_overlay(sim_heatmap, cv2.COLORMAP_MAGMA)

            # Original
            axes[i, 0].imshow(orig_img)
            axes[i, 0].set_title(f"Scan {idx+1} ({mod_str})\nTruth: {true_str}", fontweight="bold", fontsize=11)
            axes[i, 0].axis('off')

            # Grad-CAM++
            axes[i, 1].imshow(cam_overlay)
            axes[i, 1].set_title("Grad-CAM++\n(Gradient Flow)", fontweight="bold", fontsize=11)
            axes[i, 1].axis('off')

            # SSM Attention
            axes[i, 2].imshow(attn_overlay)
            axes[i, 2].set_title("SSM Attention\n(Mamba Activation)", fontweight="bold", fontsize=11)
            axes[i, 2].axis('off')

            # Prototype (With dynamic score label)
            axes[i, 3].imshow(sim_overlay)
            axes[i, 3].set_title(f"Cross-Attention\n(Similarity Score: {sim_score:.4f})", fontweight="bold", fontsize=11)
            axes[i, 3].axis('off')

            # Final Prediction
            title_color = "green" if pred_class == true_label else "red"
            axes[i, 4].imshow(sim_overlay)
            axes[i, 4].set_title(f"Pred: {pred_str} ({confidence:.1f}%)\n{match_icon}", fontweight="bold", color=title_color, fontsize=12)
            axes[i, 4].axis('off')

        plt.tight_layout()
        plt.show()
        print("--------------------------------------------------")

except NameError:
    print("System Error: Ensure model and dataloaders are defined in memory.")

# Cell 23: Clinical Visual Analytics Engine

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

print("System: Loading clinical visual analytics engine...")

def plot_project_analytics(model, data_loader, dataset_name="Unseen Test Data"):
    model.eval()
    all_probs = []
    all_labels = []

    print("--------------------------------------------------")
    print(f"System: Compiling prediction permutations for {dataset_name}...")
    print("--------------------------------------------------\n")

    # 1. Prediction Generation Phase
    with torch.no_grad():

        # Professional fixed-width progress bar
        progress_bar = tqdm(
            data_loader,
            desc=f"Scanning {dataset_name}",
            leave=False,
            bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt}'
        )

        for images, labels, modality_flags in progress_bar:
            images = images.to(device)
            modality_flags = modality_flags.to(device)

            # High-speed mixed precision inference
            with torch.amp.autocast(device_type='cuda'):
                logits, _, _ = model(images, modality_flags)

            # Convert 2D tensor into a 1D array for sklearn compatibility
            probs = torch.sigmoid(logits).detach().cpu().numpy().flatten()
            all_probs.extend(probs)

            # Safely move labels to CPU and flatten
            all_labels.extend(labels.cpu().numpy().flatten())

    # Format predictions using a strict 0.5 clinical threshold
    all_preds = (np.array(all_probs) > 0.5).astype(int)
    all_labels = np.array(all_labels)

    # 2. Canvas Initialization (3-Chart Layout)
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # CHART 1: CONFUSION MATRIX
    cm = confusion_matrix(all_labels, all_preds)

    group_names = [
        'True Negative\n(Healthy correctly identified)',
        'False Positive\n(Healthy mistaken for Tumor)',
        'False Negative\n(Tumor missed by AI)',
        'True Positive\n(Tumor correctly identified)'
    ]

    group_counts = ["{0:0.0f}".format(value) for value in cm.flatten()]
    group_percentages = ["{0:.1%}".format(value) for value in cm.flatten()/np.sum(cm)]

    labels = [f"{v1}\n\n{v2}\n({v3})" for v1, v2, v3 in zip(group_names, group_counts, group_percentages)]
    labels = np.asarray(labels).reshape(2, 2)

    sns.heatmap(
        cm, annot=labels, fmt='', cmap='Blues', ax=axes[0],
        annot_kws={"size": 11}, cbar=False, linewidths=2, linecolor='black'
    )

    axes[0].set_title(f'AI Confusion Matrix ({dataset_name})', fontsize=16, fontweight='bold', pad=15)
    axes[0].set_xlabel('AI Prediction', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Actual Ground Truth', fontsize=12, fontweight='bold')
    axes[0].xaxis.set_ticklabels(['Healthy', 'Tumor'], fontsize=11)
    axes[0].yaxis.set_ticklabels(['Healthy', 'Tumor'], fontsize=11)

    # CHART 2: RECEIVER OPERATING CHARACTERISTIC (ROC)
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    axes[1].plot(fpr, tpr, color='#ff7f0e', lw=3, label=f'Model Performance (AUC = {roc_auc:.4f})')
    axes[1].plot([0, 1], [0, 1], color='#1f77b4', lw=2, linestyle='--', label='Random Guessing (AUC = 0.50)')
    axes[1].fill_between(fpr, tpr, alpha=0.1, color='#ff7f0e')

    axes[1].set_xlim([-0.02, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12, fontweight='bold')
    axes[1].set_title(f'ROC Curve ({dataset_name})', fontsize=16, fontweight='bold', pad=15)
    axes[1].legend(loc="lower right", fontsize=12, frameon=True, shadow=True)


    # CHART 3: PRECISION-RECALL (PR) CURVE
    precision, recall, _ = precision_recall_curve(all_labels, all_probs)
    avg_precision = average_precision_score(all_labels, all_probs)

    axes[2].plot(recall, precision, color='#2ca02c', lw=3, label=f'PR Curve (AP = {avg_precision:.4f})')

    # Baseline for PR curve (ratio of positive instances)
    baseline = np.sum(all_labels) / len(all_labels)
    axes[2].plot([0, 1], [baseline, baseline], color='#d62728', lw=2, linestyle='--', label=f'Baseline ({baseline:.2f})')
    axes[2].fill_between(recall, precision, alpha=0.1, color='#2ca02c')

    axes[2].set_xlim([-0.02, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('Recall (Sensitivity)', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('Precision (Positive Predictive Value)', fontsize=12, fontweight='bold')
    axes[2].set_title(f'Precision-Recall Curve ({dataset_name})', fontsize=16, fontweight='bold', pad=15)
    axes[2].legend(loc="lower left", fontsize=12, frameon=True, shadow=True)

    plt.tight_layout()
    plt.show()

print("System: Visual analytics engine loaded successfully.")

# Cell 24: Comprehensive Visual Analytics Generation

In [ ]:
print("System: Initiating comprehensive visual analytics engine...")
print("System: Generating dashboards for both clean training and unseen testing data.\n")

try:
    # 1. Generate Training Data Dashboard (Apples-to-Apples Clean Data)
    print("System: Compiling Training Data Dashboard...")
    plot_project_analytics(eval_model, eval_train_loader, dataset_name="Training Data")

    # 2. Generate Testing Data Dashboard (Real-World Generalization)
    print("System: Compiling Unseen Testing Data Dashboard...")
    plot_project_analytics(eval_model, eval_test_loader, dataset_name="Unseen Test Data")

    print("\nSystem: Comprehensive visual analytics generation complete.")

except NameError:
    print("System Error: Required variables are missing from active memory.")
    print("System Resolution: Please ensure you ran the Aggregate Evaluation cell to load 'eval_model', 'eval_train_loader', and 'eval_test_loader' before executing this dashboard.")

# Cell 25: Installing UMAP

In [ ]:
print("System: Installing umap-learn for advanced manifold projection...")

!pip install umap-learn -q

print("System: UMAP installed successfully.")

# Cell 26: UMAP Manifold Projection

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import warnings
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

print("System: Initializing UMAP Manifold Projection...")
print("System: Extracting deep spatial features from validation set...\n")

# 1. Locate the best fold
fold_keys = list(folds_data.keys())
best_idx = np.argmax(aggregate_test["accuracy"])
best_fold = fold_keys[best_idx]

# 2. Synchronize model weights with the best performing fold
save_dir = "/content/drive/MyDrive/Tumor_Project"
fold_path = os.path.join(save_dir, f"VisionMamba_Fold_{best_fold}_Best.pth")
eval_model.load_state_dict(torch.load(fold_path, map_location=device))
eval_model.eval()

extracted_features = []
all_labels = []

# 3. Load the validation subset corresponding to the best fold
umap_loader = DataLoader(folds_data[best_fold]["val_dataset"], batch_size=32, shuffle=False)

with torch.no_grad():
    for images, labels, mod_flags in umap_loader:
        images = images.to(device)
        mod_flags = mod_flags.to(device)

        # FEATURE EXTRACTION
        logits, global_features, _ = eval_model(images, mod_flags)
        features = global_features

        extracted_features.append(features.cpu().numpy())
        all_labels.extend(labels.numpy())

# Stack lists into clean numpy arrays
extracted_features = np.vstack(extracted_features)
all_labels = np.array(all_labels)

print(f"System: Features extracted successfully. Shape: {extracted_features.shape}")
print("System: Compiling UMAP mathematical reduction (High-Dimensional : 2D)...\n")

# UMAP REDUCTION ENGINE
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.3,
    metric='euclidean',
    random_state=42,
    n_jobs=1
)
umap_embeddings = reducer.fit_transform(extracted_features)

print("System: UMAP reduction complete. Rendering visualization...")

# DATA MAPPING FOR VISUALIZATION
class_labels = ["Tumor" if label == 1 else "Healthy" for label in all_labels]

# PLOTTING
sns.set_theme(style="whitegrid", palette="muted")
fig, ax = plt.subplots(figsize=(10, 8))

# Plot:(Tumor vs Healthy)
sns.scatterplot(
    x=umap_embeddings[:, 0], y=umap_embeddings[:, 1],
    hue=class_labels,
    palette={"Tumor": "#e74c3c", "Healthy": "#2ecc71"},
    s=80, alpha=0.8, edgecolor="k", ax=ax
)
ax.set_title("UMAP Projection: Latent Space Clustering\n(Tumor vs. Healthy)", fontsize=15, fontweight="bold")
ax.set_xlabel("UMAP Dimension 1", fontsize=12)
ax.set_ylabel("UMAP Dimension 2", fontsize=12)
ax.legend(title="Diagnosis", frameon=True, fontsize=11, title_fontsize=12)

plt.tight_layout()
plt.show()

print("\nSystem: UMAP Analysis Successfully Rendered.")